In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.formula.api as smf
from pandas.io.formats.style import Styler
from statsmodels.stats.outliers_influence import variance_inflation_factor

MECHANISMS = ["m0", "m1", "m2", "m3"]
COMPONENTS = ["ifeval/prompt_accuracy", "entity_fidelity/correct", "truthfulqa/mc1"]
LABELS = {
  "ifeval/prompt_accuracy": "ifeval",
  "entity_fidelity/correct": "fidelity",
  "truthfulqa/mc1": "mc1",
  "truthfulqa/margin": "margin",
  "entity_leakage/entity_leak_rate": "leakage",
}

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 120
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 200)

In [ ]:
ev = pd.read_csv("results/evaluate.csv")
train = pd.read_csv("results/train.csv")
text_s = pd.read_csv("results/distortion.csv")

grid = ev[ev.mechanism.isin(["m1", "m2", "m3"])]
print(f"{len(ev)} evaluations, {len(train)} training runs")
ev.groupby(["dataset", "mechanism"]).size().unstack(fill_value=0)

## Coverage and sanity

In [ ]:
placebo = ev[(ev.mechanism == "m0") | ((ev.mechanism == "m2") & (ev.level == 12))]
placebo.pivot_table(index=["dataset", "model"], columns="mechanism", values=COMPONENTS).round(4)

In [ ]:
spread = (
  ev[ev.mechanism.isin(MECHANISMS)]
  .groupby(["dataset", "model", "mechanism", "level"])[COMPONENTS]
  .agg(lambda v: v.max() - v.min())
)
spread.groupby(level=["dataset", "mechanism"]).mean().round(4)

In [ ]:
(
  ev[ev.mechanism.isin(["base", "m0"])]
  .groupby(["model", "mechanism"])[COMPONENTS + ["entity_leakage/entity_leak_rate"]]
  .mean()
  .round(4)
)

## Response across levels

In [ ]:
levels = (
  grid.melt(
    id_vars=["dataset", "mechanism", "d"],
    value_vars=COMPONENTS + ["truthfulqa/margin", "entity_leakage/entity_leak_rate"],
    var_name="metric",
    value_name="value",
  )
  .assign(metric=lambda f: f.metric.map(LABELS))
  .pivot_table(index=["metric", "dataset", "mechanism"], columns="d", values="value", aggfunc="mean")
)
levels.round(4)

In [ ]:
long = grid.melt(
  id_vars=["dataset", "model", "mechanism", "d"], value_vars=COMPONENTS, var_name="metric", value_name="value"
).assign(metric=lambda f: f.metric.map(LABELS))
for dataset in sorted(long.dataset.dropna().unique()):
  g = sns.relplot(
    data=long[long.dataset == dataset],
    x="d",
    y="value",
    hue="model",
    col="metric",
    row="mechanism",
    kind="line",
    marker="o",
    errorbar="sd",
    facet_kws={"sharey": False},
    height=3.2,
    aspect=1.5,
  )
  g.set_titles("{row_name} · {col_name}")
  g.set_axis_labels("privacy-level rank (1 = strictest)", "")
  g.figure.subplots_adjust(top=0.9, hspace=0.35, wspace=0.25)
  g.figure.suptitle(dataset)
  plt.show()

## The dissociation

In [ ]:
summary = (
  ev[ev.mechanism.isin(MECHANISMS)]
  .groupby(["dataset", "mechanism"])[
    ["entity_fidelity/correct", "ifeval/prompt_accuracy", "truthfulqa/margin", "entity_leakage/entity_leak_rate"]
  ]
  .mean()
)
delta = summary.copy()
for dataset in delta.index.get_level_values(0).unique():
  delta.loc[dataset] = (delta.loc[dataset] - delta.loc[(dataset, "m0")]).to_numpy()
delta.round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, metric, title in zip(
  axes,
  ["entity_fidelity/correct", "truthfulqa/margin"],
  ["entity fidelity — M3 acts with zero text distortion", "truthfulqa margin — only the text mechanisms move it"],
):
  sns.lineplot(
    data=grid[grid.dataset == "nemotron"], x="d", y=metric, hue="mechanism", marker="o", errorbar="sd", ax=ax
  )
  ax.set_title(title)
  ax.set_xlabel("privacy-level rank (1 = strictest)")
  ax.set_ylabel(LABELS[metric])
plt.tight_layout()
plt.show()

## Identification of `c`

In [ ]:
levels = (
  grid.groupby(["dataset", "mechanism", "d"])
  .agg(S_text=("S_text", "mean"), S_gen_side=("S_gen_side", "mean"), S_gen_sd=("S_gen_side", "std"), n=("id", "size"))
  .reset_index()
)
levels.round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, column in zip(axes, ["S_text", "S_gen_side"]):
  sns.lineplot(data=levels, x="d", y=column, hue="mechanism", style="dataset", marker="o", ax=ax)
  ax.set_title(column)
  ax.set_xlabel("privacy-level rank (1 = strictest)")
plt.tight_layout()
plt.show()

In [ ]:
rows = []
for column in ["S_text", "S_gen_side"]:
  for mechanism in ["m1", "m2", "m3"]:
    sub = grid[(grid.mechanism == mechanism) & grid[column].notna() & grid.d.notna()]
    if len(sub) < 4:
      rows.append({"S": column, "mechanism": mechanism, "n": len(sub)})
      continue
    design = np.vstack([np.ones(len(sub)), sub.d.to_numpy(float)]).T
    target = sub[column].to_numpy(float)
    residual = target - design @ np.linalg.lstsq(design, target)[0]
    rms, span = float(np.sqrt((residual**2).mean())), float(np.ptp(target))
    rows.append(
      {
        "S": column,
        "mechanism": mechanism,
        "n": len(sub),
        "rms": rms,
        "range": span,
        "share": rms / span if span else np.nan,
        "constant_at": target[0] if span == 0 else np.nan,
      }
    )
pd.DataFrame(rows).round(4)

## Fit

In [ ]:
def fit(frame: pd.DataFrame, s_column: str) -> Styler:
  data = frame.dropna(subset=["Y", "d", s_column]).copy()
  data["d_z"] = (data.d - data.d.mean()) / data.d.std()
  data["S_z"] = (data[s_column] - data[s_column].mean()) / data[s_column].std()
  model = smf.mixedlm("Y ~ C(mechanism) + C(mechanism):d_z + S_z", data, groups=data["model"]).fit()
  design = data[["d_z", "S_z"]].assign(const=1.0)
  vif = {name: variance_inflation_factor(design.to_numpy(), i) for i, name in enumerate(design.columns)}
  table = pd.DataFrame({"coef": model.params, "se": model.bse, "p": model.pvalues})
  table["vif"] = [vif.get(name, np.nan) for name in table.index]
  caption = f"S = {s_column} · n = {len(data)} · mechanisms = {', '.join(sorted(data.mechanism.unique()))}"
  return table.round(4).style.set_caption(caption)

In [ ]:
fit(grid, "S_text")

In [ ]:
fit(grid, "S_gen_side")